# 从全连接层到卷积
---
## 环境配置

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import pypto
import torch
import torch_npu
import numpy as np

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
mode = pypto.RunMode.NPU

---

## 练习 6.1.1

假设卷积层(6.3)覆盖的局部区域 $\Delta = 0$。在这种情况下，证明卷积内核为每组通道独立地实现一个全连接层。

### 解答

&emsp;&emsp;局部区域 $\Delta=0$ 表示卷积核的大小等于输入的大小。实际就是问，$1×1$ 的卷积核是否等价于全连接（参见本书 $7.3$ 节：`NiN`网络结构）。因此，每个卷积核只能覆盖一个像素点，在这种情况下，卷积层的计算方式与全连接层非常相似。因为每个卷积核只能看到一个通道的信息，相当于每个卷积核只是一个全连接层的权重矩阵。 所以，卷积内核可以看作是每组通道独立地实现一个全连接层。每个卷积核都有自己的权重，每个输入通道都被独立处理，输出通道是各个输入通道的加权和。这种独立处理的方式有效地减少了权重的数量，从而降低了计算成本，并且能够提取出输入数据中的空间特征。

以下使用 `torch` 编程进行验证：

In [3]:
import torch
import torch.nn as nn

class MyNet1(nn.Module):
    def __init__(self, linear1, linear2):
        super(MyNet1, self).__init__()
        self.linear1 = linear1
        self.linear2 = linear2

    def forward(self, X):
        return self.linear2(self.linear1(nn.Flatten()(X)))

class MyNet2(nn.Module):
    def __init__(self, linear, conv2d):
        super(MyNet2, self).__init__()
        self.linear = linear
        self.conv2d = conv2d

    def forward(self, X):
        X = self.linear(nn.Flatten()(X))
        X = X.reshape(X.shape[0], -1, 1, 1)
        X = nn.Flatten()(self.conv2d(X))
        return X

linear1 = nn.Linear(15, 10)
linear2 = nn.Linear(10, 5)
conv2d = nn.Conv2d(10, 5, 1)

linear2.weight = nn.Parameter(conv2d.weight.reshape(linear2.weight.shape))
linear2.bias = nn.Parameter(conv2d.bias)

net1 = MyNet1(linear1, linear2)
net2 = MyNet2(linear1, conv2d)

X = torch.randn(2, 3, 5)

print(net1(X))
print(net2(X))

tensor([[ 0.0847,  0.2502, -0.7930, -0.1338, -0.5367],
        [-1.1455,  0.3439, -0.1890, -0.1399, -0.0573]],
       grad_fn=<AddmmBackward0>)
tensor([[ 0.0847,  0.2502, -0.7930, -0.1338, -0.5367],
        [-1.1455,  0.3439, -0.1890, -0.1399, -0.0573]],
       grad_fn=<ViewBackward0>)


使用 `PyPTO` 编程进行验证:

In [ ]:
import torch.nn as nn
from src.PyPTOConvPrimitive import corr2d

@pypto.frontend.jit(runtime_options={"run_mode": mode})
def practice_6_1_1_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    w1: pypto.Tensor([], pypto.DT_FP32),
    b1: pypto.Tensor([], pypto.DT_FP32),
    w2: pypto.Tensor([], pypto.DT_FP32),
    b2: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_cube_tile_shapes([32, 32], [64, 64], [64, 64])
    pypto.set_vec_tile_shapes(16, 16)
    x_flat = pypto.reshape(x, [x.shape[0], -1])
    linear1_out = pypto.matmul(x_flat, w1, pypto.DT_FP32) + b1
    # 1x1 卷积等价于全连接层: matmul(input_2d, weight_2d) + bias
    conv_out = pypto.matmul(linear1_out, w2, pypto.DT_FP32) + b2
    out.move(conv_out)

X = torch.randn(2, 3, 5, dtype=torch.float32, device=device)
linear1 = nn.Linear(15, 10).to(device)
conv2d = nn.Conv2d(10, 5, 1).to(device)
w1 = linear1.weight.t().contiguous()
b1 = linear1.bias.view(1, -1).contiguous()
w2 = conv2d.weight.squeeze(-1).squeeze(-1).t().contiguous()
b2 = conv2d.bias.view(1, -1).contiguous()
out = torch.zeros(2, 5, dtype=torch.float32, device=device)
practice_6_1_1_kernel(X, w1, b1, w2, b2, out)
print(f'PyPTO matmul result: {out}')

# 验证等价性: PyTorch Conv2d(1x1) 应与 matmul 结果一致
linear1_out = linear1(X.flatten(1))
conv_out_torch = conv2d(linear1_out.unsqueeze(-1).unsqueeze(-1)).flatten(1)
print(f'PyTorch Conv2d(1x1) result: {conv_out_torch}')
print(f'equivalent: {torch.allclose(out, conv_out_torch, atol=1e-3)}')

PyPTO matmul result: tensor([[-0.1952,  0.6585,  0.0091, -0.1038, -0.0412],
        [ 0.4636,  0.0911,  0.6208, -0.4578,  0.7317]], device='npu:0')
PyTorch Conv2d(1x1) result: tensor([[-0.1952,  0.6585,  0.0091, -0.1038, -0.0413],
        [ 0.4636,  0.0911,  0.6207, -0.4580,  0.7317]], device='npu:0',
       grad_fn=<ViewBackward0>)
equivalent: True


---

## 练习 6.1.2

为什么平移不变性可能也不是好主意呢？

### 解答

&emsp;&emsp;平移不变性是一种在信号处理和图像分析中常见的特性，尤其是在卷积神经网络（`CNNs`）中。它意味着一个系统或函数对输入数据的平移是不变的，即如果输入数据发生平移，输出不会改变。这对于很多应用来说是有益的，比如在图像识别中，无论物体在图像中的位置如何，模型都能识别它。\
&emsp;&emsp;平移不变性可能也存在一些局限性或不足之处:\
&emsp;&emsp;1.**丢失空间信息**：当模型对位置不敏感时，它可能无法识别对象的确切位置或对象之间的空间关系。在某些任务中，如场景理解或对象定位，这种空间信息非常重要。\
&emsp;&emsp;2.**不适用于所有任务**：对于一些特定的任务，如图像中文本的识别或布局分析，平移不变性可能不是一个理想的特性，因为这些任务需要对位置和排列非常敏感。\
&emsp;&emsp;3.**过度泛化**：平移不变性可能导致模型过度泛化，无法识别某些应该被视为不同的模式或对象。例如，在医学成像分析中，肿瘤的确切位置对于诊断至关重要。\
&emsp;&emsp;4.**缩放和旋转问题**：虽然平移不变性处理位置的变化，但它不处理缩放或旋转，这可能是图像识别中的关键因素。\
&emsp;&emsp;5.**计算效率**：为了实现平移不变性，卷积网络通常需要更多的参数和计算资源，这可能导致效率低下，尤其是在资源受限的环境中。\
&emsp;&emsp;6.**局部性限制**：平移不变性通常通过局部感受野实现，这可能限制模型捕捉长距离依赖或大尺度结构的能力。

&emsp;&emsp;参考：[https://arxiv.org/pdf/1805.12177.pdf](https://arxiv.org/pdf/1805.12177.pdf)

---

## 练习 6.1.3

当从图像边界像素获取隐藏表示时，我们需要思考哪些问题？

### 解答

&emsp;&emsp;从图像边界像素获取隐藏表示时，需要考虑一些特定的问题和挑战：\
&emsp;&emsp;1. **边界效应**：在图像边界处，像素的上下文信息可能不完整，这可能导致在提取特征时出现边界效应。边界像素没有足够的邻近像素来形成完整的局部模式，这可能影响隐藏表示的质量。\
&emsp;&emsp;2. **信息丢失**：边界像素通常不如图像中心区域的像素具有丰富的信息。在处理边界像素时，可能会丢失对整体图像理解至关重要的上下文信息。\
&emsp;&emsp;3. **尺度和旋转问题**：在边界区域，图像的尺度和旋转变化可能对隐藏表示产生更大的影响，尤其是在处理尺寸不一或旋转图像时。\
&emsp;&emsp;4. **填充策略的选择**：在使用卷积神经网络等工具时，可能需要对边界进行填充（如零填充）以保持特征图的尺寸。不同的填充策略可能会对边界像素的隐藏表示产生不同的影响。\
&emsp;&emsp;5. **噪声和伪影**：边界区域可能更容易受到图像处理过程中引入的噪声和伪影的影响，这可能会干扰隐藏表示的准确性。\
&emsp;&emsp;6. **特定应用的需求**：根据应用的不同，边界像素的处理方式可能需要调整。例如，在某些任务中，边界信息可能尤为重要，而在其他任务中，则可能不那么重要。\
&emsp;&emsp;为了有效地从图像边界像素获取隐藏表示，可能需要采用特殊的技术和策略，如使用特殊的卷积核、采用适应性边界处理方法或通过数据增强来模拟边界效应。此外，了解和评估边界像素对最终任务的影响也是很重要的。

---

## 练习 6.1.4

描述一个类似的音频卷积层的架构。

### 解答

&emsp;&emsp;一种基于卷积神经网络的音频特征生成方法，首先对声音信号进行预处理和离散傅里叶变换计算声音信号的幅度谱，形成二维谱图信号；然后搭建以上述二维谱图信号为输入的一维卷积神经网络并进行模型训练，得到特征生成器模型；最后对待测声音进行预处理和离散傅里叶变换得到二维谱图信号，并将其送入训练好的一维卷积神经网络，通过卷积网络计算，得到输出即为所要生成的音频特征，实现声音信号的音频特征生成。

&emsp;&emsp;参考：https://blog.csdn.net/david_tym/article/details/112756785

---

## 练习 6.1.5

卷积层也适合于文本数据吗？为什么？

### 解答

&emsp;&emsp;卷积层也适合于文本数据。\
&emsp;&emsp;在自然语言处理中，文本数据通常表示为词向量矩阵，其中每行代表一个词的向量表示。卷积层可以在这个矩阵上进行卷积操作，类似于图像卷积层中对图像进行卷积操作。 在卷积层中，卷积核会在输入矩阵上进行滑动窗口计算，输出一个新的特征矩阵。在文本数据中，这个特征矩阵可以看作是对输入文本的不同`n-gram`特征的提取。例如，一个大小为 $3$ 的卷积核可以提取出输入文本中每个长度为 $3$ 的`n-gram`特征。这些特征可以用于后续的分类或者回归任务。 此外，卷积层还可以与循环神经网络（`RNN`）结合使用，形成卷积神经网络（`CNN`）和循环神经网络（`RNN`）的混合模型。这种模型可以同时捕捉文本中的局部特征和全局特征，提高模型的性能。 因此，卷积层适用于文本数据，可以对文本数据进行卷积操作，提取出不同`n-gram`特征，并且可以与`RNN`结合使用，提高模型的性能。

---

## 练习 6.1.6

证明在式(6.6)中，$f * g = g * f$。

### 解答

&emsp;&emsp;通过式(6.6)的定义，我们可以得到：

$$(f * g)(x) = \int_{-\infty}^{\infty}f(y)g(x-y)dy$$

$$(g * f)(x) = \int_{-\infty}^{\infty}g(y)f(x-y)dy$$

&emsp;&emsp;要证明$f * g = g * f$，即证明：

$$\int_{-\infty}^{\infty}f(y)g(x-y)dy = \int_{-\infty}^{\infty}g(y)f(x-y)dy$$

&emsp;&emsp;为了证明上式成立，我们将其中一个积分的变量名改为$t=x-y$，则有：

$$\int_{-\infty}^{\infty}f(y)g(x-y)dy = \int_{-\infty}^{\infty}f(x-t)g(t)dt$$

&emsp;&emsp;再将这个式子代回式(6.6)中：

$$(f * g)(x) = \int_{-\infty}^{\infty}f(x-t)g(t)dt$$

&emsp;&emsp;对比式(6.6)和上面的式子，可以发现它们的形式是完全一样的，只是积分变量名不同而已。因此，我们可以得到：

$$(f * g)(x) = \int_{-\infty}^{\infty}f(y)g(x-y)dy = \int_{-\infty}^{\infty}g(y)f(x-y)dy = (g * f)(x)$$

&emsp;&emsp;因此，$f * g = g * f$，证毕。



---
## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#](https://datawhalechina.github.io/d2l-ai-solutions-manual/#)